# AI-Based Skin Disease Detection with Treatment Cost Estimation
## Phase 3: U-Net Lesion Boundary Segmentation Model Training

This Google Colab notebook trains and evaluates the lightweight **ResNet18-UNet** model on the 1,000 genuine ISIC segmentation dataset.

- **Architecture:** ResNet18 Pretrained Encoder + Progressive Bilinear Decoder + Skip Connections
- **Loss Function:** BCEWithLogitsLoss + Soft Dice Loss
- **Dataset:** 700 Train / 150 Validation / 150 Held-Out Test pairs
- **Optimization:** AdamW (lr=1e-4), ReduceLROnPlateau, Early Stopping (Patience=5)
- **Hardware:** GPU Acceleration (T4 / V100 / A100)

### Step 1: Verify GPU Environment

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
else:
    print("Warning: GPU not detected. Go to Runtime -> Change runtime type -> Select T4 GPU.")

### Step 2: Install Necessary Dependencies

In [ ]:
# Install lightweight required libraries if running on Colab
%pip install -q timm opencv-python matplotlib pandas scikit-learn

### Step 3: Extract Project Files & Set Paths
Upload `colab_unet_package.zip` into the Colab files pane, then run this cell.

In [ ]:
import os, sys, zipfile
from pathlib import Path

# Unzip if archive exists
if os.path.exists('colab_unet_package.zip') and not os.path.exists('models/unet.py'):
    print("Unzipping colab_unet_package.zip...")
    with zipfile.ZipFile('colab_unet_package.zip', 'r') as zip_ref:
        zip_ref.extractall('.')
    print("Extraction complete.")

# Ensure project root is in sys.path
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Verify folder structure
assert os.path.exists('models/unet.py'), "models/unet.py not found!"
assert os.path.exists('data/manifests/segmentation_train.csv'), "segmentation_train.csv not found!"
print("Project structure verified.")

### Step 4: Verify Segmentation DataLoaders

In [ ]:
from training.dataset import get_segmentation_dataloaders

loaders = get_segmentation_dataloaders(batch_size=8, num_workers=2)
train_loader = loaders['train']
val_loader = loaders['val']
test_loader = loaders['test']

print(f"Train samples: {len(train_loader.dataset)} (88 batches)")
print(f"Validation samples: {len(val_loader.dataset)} (19 batches)")
print(f"Held-Out Test samples: {len(test_loader.dataset)} (19 batches)")

# Inspect sample batch
for imgs, masks, ids in train_loader:
    print(f"Image batch shape: {imgs.shape} (Expected: [8, 3, 256, 256])")
    print(f"Mask batch shape:  {masks.shape} (Expected: [8, 1, 256, 256])")
    break

### Step 5: Execute U-Net Model Training
Trains ResNet18-UNet for up to 20 epochs with AdamW, mixed precision, and early stopping.

In [ ]:
from training.train_unet import run_training

history = run_training(
    epochs=20,
    batch_size=8,
    lr=1e-4,
    patience=5,
    smoke_test=False
)

### Step 6: Display Training & Validation Loss Curves

In [ ]:
from IPython.display import Image, display

print("Training vs Validation Loss Curve:")
if os.path.exists('training/results/plots/loss_curve.png'):
    display(Image('training/results/plots/loss_curve.png'))

print("Validation Dice Score Curve:")
if os.path.exists('training/results/plots/dice_curve.png'):
    display(Image('training/results/plots/dice_curve.png'))

### Step 7: View Genuine Held-Out Test Comparisons

In [ ]:
import glob

sample_files = sorted(glob.glob('training/results/samples/*.png'))
print(f"Found {len(sample_files)} sample comparison figures.")
for f in sample_files[:3]:
    display(Image(f))

### Step 8: View Final Test Evaluation Metrics

In [ ]:
import json

with open('training/results/unet_metrics.json') as f:
    metrics = json.load(f)

print("=" * 45)
print("FINAL HELD-OUT TEST EVALUATION METRICS")
print("=" * 45)
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"{k.replace('_', ' ').title():<25}: {v:.4f}")
    else:
        print(f"{k.replace('_', ' ').title():<25}: {v}")
print("=" * 45)

### Step 9: Download Trained Weights to Local Machine

In [ ]:
from google.colab import files

# Download unet_model.pth to copy back into local models/ folder
if os.path.exists('models/unet_model.pth'):
    files.download('models/unet_model.pth')
    files.download('training/results/unet_metrics.json')